# NB03 - Modern Optimizers: Lion, Sophia, ADOPT
## Background
The 2022-2024 period produced a wave of optimizers challenging AdamW. Lion (Chen et al., 2023) uses sign gradients with 2-3x less memory than Adam. Sophia (Liu et al., 2023) incorporates diagonal Hessian estimates for 2x speedup over AdamW on GPT-scale language model pretraining. ADOPT (Taniguchi et al., 2024) fixes Adam's convergence issues by reordering gradient and moment updates.

## What You'll Learn
- Lion: sign update, minimal memory footprint
- Sophia: Hutchinson's estimator for diagonal Hessian, clipped update
- ADOPT: corrected update ordering for convergence guarantees
- Benchmark across convex, non-convex, and noisy gradient problems

## Why This Matters
Knowing which optimizer to reach for depends on the problem: Lion for memory-constrained training, Sophia when Hessian estimation is feasible, ADOPT when AdamW diverges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Callable

@dataclass
class Lion:
    lr: float = 1e-4
    beta1: float = 0.9
    beta2: float = 0.99
    weight_decay: float = 0.0
    _m: Optional[np.ndarray] = field(default=None, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._m is None:
            self._m = np.zeros_like(params)
        update = np.sign(self.beta1 * self._m + (1 - self.beta1) * grads)
        self._m = self.beta2 * self._m + (1 - self.beta2) * grads
        return params * (1 - self.lr * self.weight_decay) - self.lr * update

    def reset(self): self._m = None

@dataclass
class Sophia:
    lr: float = 1e-3
    beta1: float = 0.965
    beta2: float = 0.99
    eps: float = 1e-12
    rho: float = 0.04
    weight_decay: float = 0.0
    hessian_update_freq: int = 10
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _h: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray,
             fn: Optional[Callable] = None) -> np.ndarray:
        if self._m is None:
            self._m = np.zeros_like(params)
            self._h = np.ones_like(params)
        self._t += 1
        self._m = self.beta1 * self._m + (1 - self.beta1) * grads
        if self._t % self.hessian_update_freq == 0 and fn is not None:
            z = np.random.choice([-1.0, 1.0], size=params.shape)
            eps = 1e-5
            _, g1 = fn(params + eps * z)
            _, g2 = fn(params - eps * z)
            hess_diag = z * (g1 - g2) / (2 * eps)
            self._h = self.beta2 * self._h + (1 - self.beta2) * hess_diag**2
        update = np.clip(self._m / (self._h + self.eps), -self.rho, self.rho)
        return params * (1 - self.lr * self.weight_decay) - self.lr * update

    def reset(self): self._m = self._h = None; self._t = 0

@dataclass
class ADOPT:
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.9999
    eps: float = 1e-6
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _v: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._m is None:
            self._m = grads.copy()
            self._v = grads**2
            self._t = 1
            return params - self.lr * grads / (np.sqrt(self._v) + self.eps)
        self._t += 1
        self._v = self.beta2 * self._v + (1 - self.beta2) * grads**2
        g_norm = grads / (np.sqrt(self._v) + self.eps)
        self._m = self.beta1 * self._m + (1 - self.beta1) * g_norm
        return params - self.lr * self._m

    def reset(self): self._m = self._v = None; self._t = 0

@dataclass
class AdamW:
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.999
    eps: float = 1e-8
    weight_decay: float = 0.01
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _v: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._m is None: self._m = np.zeros_like(params); self._v = np.zeros_like(params)
        self._t += 1
        self._m = self.beta1*self._m + (1-self.beta1)*grads
        self._v = self.beta2*self._v + (1-self.beta2)*grads**2
        m_hat = self._m/(1-self.beta1**self._t)
        v_hat = self._v/(1-self.beta2**self._t)
        return params*(1-self.lr*self.weight_decay) - self.lr*m_hat/(np.sqrt(v_hat)+self.eps)

    def reset(self): self._m = self._v = None; self._t = 0

print('Lion, Sophia, ADOPT, AdamW implemented.')

In [ ]:
def quadratic(x):
    return x[0]**2 + 10*x[1]**2, np.array([2*x[0], 20*x[1]])

def noisy_quadratic(x, noise=0.5):
    loss, grad = quadratic(x)
    return loss, grad + noise * np.random.randn(*grad.shape)

def run_opt(opt, fn, x0, steps=500, sophia_fn=None):
    x = x0.copy().astype(float)
    opt.reset()
    losses = []
    for _ in range(steps):
        loss, grad = fn(x)
        losses.append(loss)
        if isinstance(opt, Sophia) and sophia_fn is not None:
            x = opt.step(x, grad, fn=sophia_fn)
        else:
            x = opt.step(x, grad)
    return losses

np.random.seed(42)
x0 = np.array([-1.5, 0.8])
optimizers = [
    ('AdamW',  AdamW(lr=1e-2, weight_decay=0.01)),
    ('Lion',   Lion(lr=1e-3, weight_decay=0.01)),
    ('Sophia', Sophia(lr=5e-3, weight_decay=0.01)),
    ('ADOPT',  ADOPT(lr=1e-2)),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (pname, fn) in zip(axes, [('Quadratic', quadratic), ('Noisy Quadratic', noisy_quadratic)]):
    for label, opt in optimizers:
        losses = run_opt(opt, fn, x0, steps=500,
                         sophia_fn=(quadratic if pname == 'Quadratic' else None))
        ax.semilogy([max(l, 1e-10) for l in losses], label=label)
    ax.set_title(f'Optimizer Comparison: {pname}')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Loss (log scale)')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_03_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()
print('Benchmark complete.')

In [ ]:
print('=== Optimizer Memory Footprint at 7B Scale ===')
param_count = 7_000_000_000
gb = 1024**3
memory = {
    'SGD (no momentum)': 1,
    'SGD+Momentum': 2,
    'Adam/AdamW': 3,
    'Lion': 2,
    'Sophia': 3,
    'ADOPT': 3,
}
print(f'{'Optimizer':25s} | {'State tensors':>14} | {'Total FP32 GB':>15}')
print('-' * 60)
for name, n in memory.items():
    total_gb = n * param_count * 4 / gb
    print(f'{name:25s} | {n:>14} | {total_gb:>15.1f}')
print()
print('Lion saves ~33% optimizer state vs Adam - meaningful at 70B+ scale')